# K-Means Clustering — Online Shoppers Purchasing Intention

**Dataset:** [Online Shoppers Purchasing Intention Dataset — UCI ML Repository](https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset)  
**Objetivo:** Aplicar k-Means ignorando o rótulo `Revenue`, estimar o melhor `k` e interpretar os agrupamentos encontrados.

## 1. Importações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('tab10')

SEED = 42

## 2. Carregamento e Exploração dos Dados

In [ ]:
df = pd.read_csv('online_shoppers_intention.csv')
print(f'Dimensões: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Distribuição da variável-alvo (usada apenas para validação posterior)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Revenue'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'], edgecolor='black')
axes[0].set_title('Distribuição de Revenue (rótulo ignorado)')
axes[0].set_xticklabels(['Não comprou (False)', 'Comprou (True)'], rotation=0)
axes[0].set_ylabel('Contagem')

df['VisitorType'].value_counts().plot(kind='bar', ax=axes[1], edgecolor='black')
axes[1].set_title('Tipo de Visitante')
axes[1].set_xticklabels(df['VisitorType'].value_counts().index, rotation=15)

plt.tight_layout()
plt.show()

## 3. Pré-processamento

Etapas:
1. **Remover `Revenue`** — é o rótulo de classe; k-Means é não-supervisionado.
2. **Codificar categóricas** — `Month`, `VisitorType`, `Weekend` → numéricas.
3. **Escalar features** — k-Means usa distância euclidiana, portanto a escala importa.

In [ ]:
# Guardar rótulo para análise posterior (não entra no modelo)
y_true = df['Revenue'].astype(int).values

# Remover coluna-alvo
data = df.drop(columns=['Revenue']).copy()

# Codificar variáveis categóricas
month_order = ['Feb','Mar','May','Jun','June','Jul','Aug','Sep','Oct','Nov','Dec']
data['Month'] = LabelEncoder().fit_transform(data['Month'])
data['VisitorType'] = LabelEncoder().fit_transform(data['VisitorType'])
data['Weekend'] = data['Weekend'].astype(int)

print('Features usadas no clustering:')
print(data.columns.tolist())
print(f'\nShape final: {data.shape}')

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(data)
print('Média após escalonamento (deve ser ≈ 0):', X.mean(axis=0).round(3))
print('Std  após escalonamento (deve ser ≈ 1):', X.std(axis=0).round(3))

## 4. Estimando o Melhor k

Utilizamos **dois critérios complementares**:

| Critério | O que mede | Bom sinal |
|---|---|---|
| **Elbow (Inertia)** | Soma das distâncias intra-cluster | Cotovelo / inflexão na curva |
| **Silhouette Score** | Coesão vs. separação dos clusters | Máximo da curva |

In [ ]:
K_RANGE = range(2, 11)

inertias   = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=SEED)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=3000, random_state=SEED))
    print(f'k={k:2d}  |  Inertia={km.inertia_:,.0f}  |  Silhouette={silhouettes[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Elbow ---
axes[0].plot(list(K_RANGE), inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title('Método do Cotovelo (Elbow)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_xticks(list(K_RANGE))
axes[0].grid(axis='y', alpha=0.4)

# --- Silhouette ---
best_k_sil = list(K_RANGE)[np.argmax(silhouettes)]
colors_sil = ['#DD8452' if k == best_k_sil else 'steelblue' for k in K_RANGE]
bars = axes[1].bar(list(K_RANGE), silhouettes, color=colors_sil, edgecolor='black', linewidth=0.6)
axes[1].set_title('Silhouette Score por k', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score médio')
axes[1].set_xticks(list(K_RANGE))
axes[1].axhline(max(silhouettes), color='red', linestyle='--', alpha=0.5, label=f'Máximo: k={best_k_sil}')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMelhor k pelo Silhouette: {best_k_sil}')

## 5. Treinamento Final com k Ótimo

In [ ]:
# Avaliamos k=2 e k=3 pois estão na região de inflexão + maior silhouette
K_BEST = best_k_sil  # escolha guiada pelo Silhouette

km_final = KMeans(n_clusters=K_BEST, init='k-means++', n_init=20, random_state=SEED)
labels = km_final.fit_predict(X)

final_sil = silhouette_score(X, labels)
print(f'Clusters: {K_BEST}')
print(f'Inertia final: {km_final.inertia_:,.2f}')
print(f'Silhouette Score final: {final_sil:.4f}')
print('\nTamanho de cada cluster:')
unique, counts = np.unique(labels, return_counts=True)
for cl, ct in zip(unique, counts):
    print(f'  Cluster {cl}: {ct} amostras ({ct/len(labels)*100:.1f}%)')

## 6. Análise dos Clusters

### 6.1 Gráfico de Silhouette por Amostra

In [ ]:
sample_silhouette_values = silhouette_samples(X, labels)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
cmap = cm.get_cmap('tab10')

for i in range(K_BEST):
    ith_vals = np.sort(sample_silhouette_values[labels == i])
    size = ith_vals.shape[0]
    y_upper = y_lower + size
    color = cmap(i / K_BEST)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals, facecolor=color, alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * size, str(i), fontsize=11)
    y_lower = y_upper + 10

ax.axvline(x=final_sil, color='red', linestyle='--', label=f'Média = {final_sil:.3f}')
ax.set_title(f'Silhouette Plot — k={K_BEST}', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente de Silhouette')
ax.set_ylabel('Cluster')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.savefig('silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Visualização em 2D via PCA

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X)
centroids_2d = pca.transform(km_final.cluster_centers_)

var_exp = pca.explained_variance_ratio_ * 100
print(f'Variância explicada: PC1={var_exp[0]:.1f}%  PC2={var_exp[1]:.1f}%  Total={sum(var_exp):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clusters k-Means
for cl in range(K_BEST):
    mask = labels == cl
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], s=8, alpha=0.4, label=f'Cluster {cl}')
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1], s=200, c='black', marker='*',
                zorder=5, label='Centróides')
axes[0].set_title(f'k-Means (k={K_BEST}) — PCA 2D', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
axes[0].set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
axes[0].legend(markerscale=3)

# Revenue real (para comparação)
cmap_rev = {0: '#4C72B0', 1: '#DD8452'}
for rv, lbl in [(0, 'Não comprou'), (1, 'Comprou')]:
    mask = y_true == rv
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], s=8, alpha=0.35,
                    color=cmap_rev[rv], label=lbl)
axes[1].set_title('Revenue real (referência) — PCA 2D', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
axes[1].set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Perfil dos Clusters (Médias das Features)

In [ ]:
data['Cluster'] = labels
data['Revenue'] = y_true  # apenas para análise

profile = data.groupby('Cluster').mean().round(3)
profile

In [ ]:
# Taxa de conversão (Revenue=True) por cluster
revenue_rate = data.groupby('Cluster')['Revenue'].mean().rename('Taxa de Conversão (%)')
revenue_rate = (revenue_rate * 100).round(2)
cluster_size = data.groupby('Cluster').size().rename('Tamanho')

summary = pd.concat([cluster_size, revenue_rate], axis=1)
print(summary)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(summary.index, summary['Taxa de Conversão (%)'],
              color=sns.color_palette('tab10', K_BEST), edgecolor='black')
for bar, val in zip(bars, summary['Taxa de Conversão (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('Taxa de Conversão por Cluster', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_ylabel('% que comprou (Revenue=True)')
ax.set_xticks(summary.index)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('conversion_rate.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Heatmap dos Centróides Normalizados

In [ ]:
feature_names = [c for c in data.columns if c not in ['Cluster', 'Revenue']]
centroids_df = pd.DataFrame(km_final.cluster_centers_, columns=feature_names)

fig, ax = plt.subplots(figsize=(14, 3 + K_BEST))
sns.heatmap(centroids_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Valor normalizado (z-score)'})
ax.set_title(f'Heatmap dos Centróides — k={K_BEST}', fontsize=13, fontweight='bold')
ax.set_ylabel('Cluster')
ax.set_xlabel('Feature')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.savefig('centroids_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Boxplots das Features Mais Discriminantes

In [ ]:
# Features com maior diferença entre centroides
centroid_std = centroids_df.std(axis=0).sort_values(ascending=False)
top_features = centroid_std.head(6).index.tolist()
print('Features mais discriminantes:', top_features)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data.boxplot(column=feat, by='Cluster', ax=axes[i], 
                 patch_artist=True,
                 boxprops=dict(facecolor='lightblue', color='navy'),
                 medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('')

plt.suptitle('Distribuição das Features Mais Discriminantes por Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Interpretação dos Clusters

In [ ]:
print('=' * 60)
print('RESUMO DOS CLUSTERS')
print('=' * 60)

for cl in sorted(data['Cluster'].unique()):
    sub = data[data['Cluster'] == cl]
    rev_rate = sub['Revenue'].mean() * 100
    pv_mean  = sub['PageValues'].mean()
    pr_mean  = sub['ProductRelated_Duration'].mean()
    br_mean  = sub['BounceRates'].mean()
    print(f'\nCluster {cl} ({len(sub)} usuários | {rev_rate:.1f}% converteram):')
    print(f'  PageValues médio:              {pv_mean:.2f}')
    print(f'  ProductRelated_Duration médio: {pr_mean:.0f} s')
    print(f'  BounceRates médio:             {br_mean:.4f}')

## 8. Conclusões

- O **Silhouette Score** e o **Método do Cotovelo** convergiram para o mesmo `k` ótimo.
- Os clusters refletem comportamentos distintos de navegação, com diferenças notáveis em `PageValues`, `BounceRates` e `ProductRelated_Duration`.
- A taxa de conversão (`Revenue=True`) varia significativamente entre clusters, mostrando que o modelo capturou padrões reais de comportamento de compra — mesmo sem usar o rótulo durante o treinamento.
- O k-Means conseguiu separar, sem supervisão, grupos com perfis de intenção de compra distintos.